In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

try:
    from IPython.display import display
except ImportError:
    # Якщо IPython недоступний, використовується стандартний print
    display = print


warnings.filterwarnings("ignore")



# Основні налаштування файлів
# які використовуються для 
# формування фінального датасету


# Базова директорія з обробленими даними
DATA_PROCESSED = Path(r"C:\programas\diploma\data\data_processed")

# Вхідні файли після попереднього feature engineering
TRAIN_INPUT_NAME = "final_train_fe_v2.csv"
TEST_INPUT_NAME = "final_test_fe_v2.csv"

# Кількість найважливіших ознак, які залишаються у фінальному датасеті
TOP_K = 200

# Назви вихідних файлів із відфільтрованими ознаками
TRAIN_OUTPUT_NAME = f"final_train_clean_top{TOP_K}.csv"
TEST_OUTPUT_NAME = f"final_test_clean_top{TOP_K}.csv"

# Назви цільової змінної та унікального ідентифікатора клієнта
TARGET_COL = "TARGET"
ID_COL = "SK_ID_CURR"

# Фіксований seed для відтворюваності результатів
RANDOM_STATE = 42
# Кількість фолдів для стратифікованої крос-валідації
N_SPLITS = 5


# Функція знаходить 
# потрібний файл у 
# стандартних папках проєкту

def find_file(file_name):
    # Перелік кандидатів шляхів — від абсолютного до відносних рівнів вгору
    # Порядок пошуку: спочатку абсолютний шлях, потім DATA_PROCESSED, потім відносні
    possible_paths = [
        Path(file_name),
        DATA_PROCESSED / file_name,
        Path("data_processed") / file_name,
        Path("data/data_processed") / file_name,
        Path("../data_processed") / file_name,
        Path("../data/data_processed") / file_name,
        Path("../../data/data_processed") / file_name
    ]

    # Перевіряємо кожен шлях 
    # по черзі — повертаємо перший, що існує
    for path in possible_paths:
        if path.exists():
            return path

    # Якщо жоден з явних шляхів 
    # не підійшов — рекурсивний пошук по всьому проєкту
    # rglob("*") обходить усі 
    # підкаталоги, що може бути повільним на великих ФС
    found_files = list(Path.cwd().rglob(file_name))

    if len(found_files) > 0:
        # Береться перший знайдений файл, якщо їх декілька
        return found_files[0]

    raise FileNotFoundError(f"Файл {file_name} не знайдено.")


# Функція зчитує train і test 
# набори після попереднього 
# feature engineering

def load_feature_engineered_data():
    # Пошук файлів у файловій системі проєкту
    train_path = find_file(TRAIN_INPUT_NAME)
    test_path = find_file(TEST_INPUT_NAME)

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Перевірка наявності обов'язкових колонок у train
    if TARGET_COL not in train_df.columns:
        raise ValueError(f"У train-файлі немає цільової змінної {TARGET_COL}.")

    if ID_COL not in train_df.columns:
        raise ValueError(f"У train-файлі немає ідентифікатора {ID_COL}.")

    # Перевірка наявності ідентифікатора у test 
    if ID_COL not in test_df.columns:
        raise ValueError(f"У test-файлі немає ідентифікатора {ID_COL}.")

    print("Вхідні дані завантажено")
    print("Train file:", train_path)
    print("Test file :", test_path)
    print("Train shape:", train_df.shape)
    print("Test shape :", test_df.shape)

    return train_df, test_df, train_path, test_path


# Функція формує 
# матрицю ознак X 
# та цільову змінну y

def prepare_xy(train_df):
    # Цільова змінна приводиться 
    # до цілочисельного типу (0/1)
    y = train_df[TARGET_COL].astype(int)

    # Видаляємо службові колонки, які не є ознаками
    # errors="ignore" захищає від KeyError,
    #  якщо колонка вже відсутня
    drop_cols = [TARGET_COL, ID_COL]
    X = train_df.drop(columns=drop_cols, errors="ignore")

    # Нескінченні значення замінюються на пропуски
    # щоб модель не отримувала некоректні числа
    # CatBoost допускає NaN, але не inf
    X = X.replace([np.inf, -np.inf], np.nan)

    print("\nМатрицю ознак підготовлено")
    print("X shape:", X.shape)
    # Частка позитивних випадків — важливий показник балансу класів
    print("Target rate:", round(float(y.mean()), 4))

    return X, y


# Функція створює модель CatBoost
# для оцінювання важливості ознак

def build_catboost_model():
    model = CatBoostClassifier(
        # Велика кількість ітерацій 
        # компенсується раннім зупиненням (od_wait)
        iterations=60000,
        # Невелика learning rate дозволяє 
        # будувати більш узагальнені дерева
        learning_rate=0.02,
        depth=7,
        # L2-регуляризація листів зменшує 
        # перенавчання на малих вибірках
        l2_leaf_reg=12,
        # Випадковість у силі розщеплень — додатковий засіб регуляризації
        random_strength=2.0,
        # Bagging temperature керує 
        # розподілом Пуассона для bootstrap-ваг
        bagging_temperature=0.6,
        # rsm — частка ознак, що випадково 
        # відбираються для кожного дерева
        rsm=0.7,
        # Мінімальна кількість зразків 
        # у листі обмежує переспеціалізацію
        min_data_in_leaf=60,
        loss_function="Logloss",
        eval_metric="AUC",
        # Автоматичне балансування класів 
        # важливе при сильному дисбалансі класів
        auto_class_weights="Balanced",
        # Раннє зупинення за кількістю ітерацій без покращення
        od_type="Iter",
        od_wait=900,
        random_seed=RANDOM_STATE,
        # verbose=False вимикає вивід
        # прогресу навчання у консоль
        verbose=False
    )

    return model


# Функція обчислює out-of-fold 
# прогнози CatBoost для оцінювання якості ознак

def calculate_oof_catboost(X, y):
    # Стратифікація зберігає пропорцію класів у кожному фолді
    # shuffle=True необхідний при shuffle=False 
    # за замовчуванням у pandas read_csv
    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    # Масив для накопичення прогнозів 
    # поза навчальною вибіркою
    oof = np.zeros(len(X), dtype=float)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        # Розбиваємо дані на навчальну 
        # та валідаційну частини поточного фолду
        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = build_catboost_model()

        # eval_set передається для 
        # моніторингу та раннього зупинення
        model.fit(
            X_train,
            y_train,
            eval_set=(X_valid, y_valid),
            use_best_model=True
        )

        # Беремо ймовірність позитивного класу 
        # predict_proba повертає 
        # масив [prob_0, prob_1] для кожного зразка
        fold_prediction = model.predict_proba(X_valid)[:, 1]
        oof[valid_idx] = fold_prediction

        fold_roc = roc_auc_score(y_valid, fold_prediction)
        print(f"Fold {fold}: ROC-AUC = {fold_roc:.5f}")

    # Загальна якість OOF-прогнозів по всьому train
    # roc_auc_score обчислюється на 
    # повному масиві oof, а не по середньому фолдів
    roc = roc_auc_score(y, oof)
    # PR-AUC більш чутливий до якості
    # моделі при дисбалансі класів
    pr = average_precision_score(y, oof)

    print("\nOOF ROC-AUC:", round(roc, 6))
    print("OOF PR-AUC :", round(pr, 6))

    return oof


# Функція обчислює важливість 
# ознак на основі CatBoost у режимі крос-валідації

def calculate_feature_importance(X, y):
    # Та сама стратегія крос-валідації, що й 
    # для OOF, для порівнянності результатів
    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    # Масив важливостей ініціалізується 
    # нулями значення сумуються по фолдах
    importance = np.zeros(X.shape[1], dtype=float)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        # Розбиваємо дані на навчальну та 
        # валідаційну частини поточного фолду
        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = build_catboost_model()

        # eval_set передається для раннього 
        # зупинення — use_best_model вибирає найкращу ітерацію
        model.fit(
            X_train,
            y_train,
            eval_set=(X_valid, y_valid),
            use_best_model=True
        )

        # PredictionValuesChange показує 
        # внесок кожної ознаки у прогнози моделі
        # Це швидший та інтерпретованіший 
        # метод порівняно з permutation importance
        fold_importance = model.get_feature_importance(type="PredictionValuesChange")
        importance += fold_importance

        print(f"Fold {fold}: best iteration = {model.get_best_iteration()}")

    # Усереднення по фолдах дає стабільнішу оцінку важливості
    importance = importance / N_SPLITS

    # Формуємо таблицю: назва ознаки + усереднена важливість
    # X.columns зберігає оригінальний порядок ознак
    importance_table = pd.DataFrame({
        "feature": X.columns,
        "importance": importance
    })

    # Сортування від найважливіших до найменш важливих
    # reset_index забезпечує послідовну нумерацію після сортування
    importance_table = importance_table.sort_values(
        by="importance",
        ascending=False
    ).reset_index(drop=True)

    print("\nТоп-20 найважливіших ознак:")
    display(importance_table.head(20))

    return importance_table



# Функція вибирає TOP_K найбільш інформативних ознак


def select_top_features(importance_table, top_k):
    # Беремо перші top_k рядків
    # із вже відсортованої таблиці важливостей
    # .tolist() перетворює pandas Series 
    # у звичайний список Python для подальшого використання
    selected_features = importance_table["feature"].head(top_k).tolist()

    print(f"\nВідібрано {len(selected_features)} ознак для фінального набору даних")

    return selected_features



# Функція перевіряє, чи всі 
# відібрані ознаки є у train і test наборах


def check_selected_features(train_df, test_df, selected_features):
    # Відібрані ознаки мають бути
    # присутні в обох датасетах, інакше предбачення неможливе
    # List comprehension будує список
    # відсутніх колонок для зручного діагностичного повідомлення
    missing_train = [col for col in selected_features if col not in train_df.columns]
    missing_test = [col for col in selected_features if col not in test_df.columns]

    if missing_train or missing_test:
        raise KeyError(
            "Не всі відібрані ознаки знайдено у вхідних файлах. "
            f"Train missing: {missing_train[:10]} | Test missing: {missing_test[:10]}"
        )

    print("Перевірку наявності відібраних ознак виконано успішно")


# Функція формує фінальні train і test набори з ознаками
def build_final_datasets(train_df, test_df, selected_features):
    # У train зберігаємо ID, TARGET та відібрані ознаки
    # .copy() запобігає SettingWithCopyWarning при подальших змінах датафрейму
    train_clean = train_df[[ID_COL, TARGET_COL] + selected_features].copy()
    # У test TARGET відсутній — лише ID та ознаки для прогнозування
    test_clean = test_df[[ID_COL] + selected_features].copy()

    # Нескінченні значення прибираються, а пропуски 
    # залишаються для подальшої обробки моделями
    # Виконується окремо для train та test, щоб 
    # уникнути витоку інформації між наборами
    train_clean = train_clean.replace([np.inf, -np.inf], np.nan)
    test_clean = test_clean.replace([np.inf, -np.inf], np.nan)

    print("\nФінальні набори даних сформовано")
    print("Train clean shape:", train_clean.shape)
    print("Test clean shape :", test_clean.shape)

    return train_clean, test_clean


# Функція зберігає фінальні датасети і таблицю важливості ознак
def save_results(train_clean, test_clean, importance_table, train_path):
    # Зберігаємо всі артефакти поряд із вхідними файлами для зручності
    # .parent повертає директорію, де знаходиться train-файл
    output_dir = train_path.parent

    train_output_path = output_dir / TRAIN_OUTPUT_NAME
    test_output_path = output_dir / TEST_OUTPUT_NAME
    # Таблиця важливостей зберігається для подальшого аналізу та документування
    importance_output_path = output_dir / f"feature_importance_top{TOP_K}.csv"

    # utf-8-sig додає BOM, щоб Excel коректно відкривав кирилицю
    train_clean.to_csv(train_output_path, index=False, encoding="utf-8-sig")
    test_clean.to_csv(test_output_path, index=False, encoding="utf-8-sig")
    importance_table.to_csv(importance_output_path, index=False, encoding="utf-8-sig")

    print("\nФайли збережено:")
    print(train_output_path)
    print(test_output_path)
    print(importance_output_path)


# Функція перевіряє якість фінального набору ознак на CatBoost
def evaluate_final_top_features(train_clean):
    # Підготовка даних аналогічна prepare_xy, але вже на скороченому наборі ознак
    y = train_clean[TARGET_COL].astype(int)
    X = train_clean.drop(columns=[TARGET_COL, ID_COL], errors="ignore")
    X = X.replace([np.inf, -np.inf], np.nan)

    print(f"\nПеревірка якості фінального набору TOP-{TOP_K}")
    # OOF-оцінка дозволяє порівняти якість до і після відбору ознак
    calculate_oof_catboost(X, y)


# Функція запускає всі етапи формування фінальних даних
def main():
    # Крок 1: завантаження вхідних даних після feature engineering
    # Повертає також шляхи до файлів для визначення директорії збереження
    train_df, test_df, train_path, _ = load_feature_engineered_data()

    # Крок 2: формування матриці ознак та цільової змінної
    X, y = prepare_xy(train_df)

    # Крок 3: обчислення важливості ознак через крос-валідацію CatBoost
    # Це найтриваліший етап — N_SPLITS повних навчань моделі
    importance_table = calculate_feature_importance(X, y)

    # Крок 4: відбір TOP_K найважливіших ознак
    # TOP_K=200 — компроміс між повнотою інформації та швидкістю навчання
    selected_features = select_top_features(importance_table, TOP_K)

    # Крок 5: перевірка, що всі ознаки є і в train, і в test
    # Важливо зробити до побудови датасетів, щоб не зберігати неповні файли
    check_selected_features(train_df, test_df, selected_features)

    # Крок 6: побудова фінальних датасетів із відібраними ознаками
    train_clean, test_clean = build_final_datasets(
        train_df=train_df,
        test_df=test_df,
        selected_features=selected_features
    )

    # Крок 7: збереження результатів на диск
    save_results(
        train_clean=train_clean,
        test_clean=test_clean,
        importance_table=importance_table,
        train_path=train_path
    )

    # Крок 8: фінальна OOF-оцінка якості скороченого набору ознак
    # Дозволяє порівняти метрики до (повний набір) і після (TOP_K) відбору
    evaluate_final_top_features(train_clean)

    print("\nФормування фінальних даних завершено.")


# Запуск формування final_train_clean_top200.csv і final_test_clean_top200.csv
main()